In [1]:
# Cell 1: bootstrap, local model config, and browser profile.
from __future__ import annotations

import json
import os
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

PROJECT_ROOT = Path(r"D:\_Desktop\Projects\Automations prj\Job_search")
CHROME_EXE = Path(os.environ.get("BROWSER_USE_CHROME_EXE", r"C:\Program Files\Google\Chrome\Application\chrome.exe"))
CHROME_USER_DATA_DIR = Path(os.environ.get("BROWSER_USE_CHROME_USER_DATA_DIR", r"D:\_Desktop\Projects\Automations prj\User Data"))
CHROME_USER_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_USER_DATA_DIR = CHROME_USER_DATA_DIR
CDP_PORT = int(os.environ.get("BROWSER_USE_CDP_PORT", "9222"))
LLM_BACKEND = "openrouter"
LLAMA_BASE_URL = os.environ.get("BROWSER_USE_LLM_BASE_URL", "http://127.0.0.1:8080/v1")
LLAMA_MODEL_FALLBACK = os.environ.get("BROWSER_USE_LLM_MODEL", "Qwen3.5-9B.Q4_K_M.gguf")
LLAMA_API_KEY = os.environ.get("BROWSER_USE_LLM_API_KEY", "sk-local")
OPENROUTER_BASE_URL = os.environ.get("BROWSER_USE_OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.environ.get("BROWSER_USE_OPENROUTER_MODEL", "qwen/qwen3-30b-a3b-instruct-2507")
OPENROUTER_API_KEY_PATH = Path(os.environ.get("BROWSER_USE_OPENROUTER_API_KEY_PATH", r"D:\_Desktop\api_key_openrouter.txt"))
MAX_HISTORY_ITEMS = 8
FLASH_MODE = True
USE_VISION = False


def _load_json(url: str, timeout: int = 10) -> dict:
    request = urllib.request.Request(url, headers={"Accept": "application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def resolve_local_model_name() -> str:
    try:
        payload = _load_json(f"{LLAMA_BASE_URL.rstrip('/')}/models")
        models = payload.get("data") or []
        if models and isinstance(models, list):
            first = models[0]
            if isinstance(first, dict) and first.get("id"):
                return str(first["id"])
    except Exception as exc:
        print(f"Model probe failed, using fallback: {exc}")
    return LLAMA_MODEL_FALLBACK


def resolve_llm_runtime() -> dict:
    backend = LLM_BACKEND
    if backend == "openrouter":
        if not OPENROUTER_API_KEY_PATH.exists():
            raise RuntimeError(f"OpenRouter API key file not found: {OPENROUTER_API_KEY_PATH}")
        api_key = OPENROUTER_API_KEY_PATH.read_text(encoding="utf-8").strip()
        if not api_key:
            raise RuntimeError(f"OpenRouter API key file is empty: {OPENROUTER_API_KEY_PATH}")
        return {
            "backend": backend,
            "model": OPENROUTER_MODEL,
            "base_url": OPENROUTER_BASE_URL,
            "api_key": api_key,
        }
    return {
        "backend": "local",
        "model": resolve_local_model_name(),
        "base_url": LLAMA_BASE_URL,
        "api_key": LLAMA_API_KEY,
    }


LLM_RUNTIME = resolve_llm_runtime()
MODEL_NAME = LLM_RUNTIME["model"]
LLM_BASE_URL = LLM_RUNTIME["base_url"]
LLM_API_KEY = LLM_RUNTIME["api_key"]

browser_process = globals().get("browser_process")
if browser_process is not None:
    try:
        if browser_process.poll() is None:
            browser_process.terminate()
            try:
                browser_process.wait(timeout=10)
            except Exception:
                browser_process.kill()
    except Exception:
        pass


def _kill_stale_chrome_instances() -> None:
    try:
        safe_dir = str(CHROME_USER_DATA_DIR)
        command = (
            "$targets = Get-CimInstance Win32_Process | Where-Object { $_.Name -eq 'chrome.exe' -and "
            "((($_.CommandLine -like '*--remote-debugging-port=" + str(CDP_PORT) + "*') -or ($_.CommandLine -like '*" + safe_dir + "*'))) }; "
            "$targets | ForEach-Object { Stop-Process -Id $_.ProcessId -Force }"
        )
        subprocess.run(["powershell", "-NoProfile", "-Command", command], check=False, capture_output=True)
    except Exception:
        pass


_kill_stale_chrome_instances()

if not CHROME_EXE.exists():
    raise RuntimeError(f'Chrome not found: {CHROME_EXE}')

browser_process = subprocess.Popen(
    [
        str(CHROME_EXE),
        f'--remote-debugging-port={CDP_PORT}',
        f'--user-data-dir={LOCAL_USER_DATA_DIR}',
        '--new-window',
        '--start-maximized',
        '--window-position=0,0',
        '--window-size=1440,1080',
        '--no-first-run',
        '--no-default-browser-check',
        'about:blank',
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)

BROWSER_CDP_URL = f'http://127.0.0.1:{CDP_PORT}'

print(json.dumps({
    "project_root": str(PROJECT_ROOT),
    "user_data_dir": str(CHROME_USER_DATA_DIR),
    "chrome_exe": str(CHROME_EXE),
    "cdp_url": BROWSER_CDP_URL,
    "llm_backend": LLM_BACKEND,
    "llm_base_url": LLM_BASE_URL,
    "model": MODEL_NAME,
    "max_history_items": MAX_HISTORY_ITEMS,
    "flash_mode": FLASH_MODE,
    "use_vision": USE_VISION,
}, indent=2))


{
  "project_root": "D:\\_Desktop\\Projects\\Automations prj\\Job_search",
  "user_data_dir": "D:\\_Desktop\\Projects\\Automations prj\\Job_search\\runtime\\browser_use_oss_profile",
  "chrome_exe": "C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe",
  "cdp_url": "http://127.0.0.1:9222",
  "llama_base_url": "http://127.0.0.1:8080/v1",
  "model": "Qwen3.5-9B.Q4_K_M.gguf",
  "max_history_items": 8,
  "flash_mode": true,
  "use_vision": false
}


In [5]:
# Cell 2: Browser Use agent setup and runner.
from browser_use import Agent, BrowserProfile, BrowserSession
from browser_use.llm.openai.chat import ChatOpenAI

browser_profile = BrowserProfile(
    cdp_url=BROWSER_CDP_URL,
    is_local=False,
    headless=False,
    window_size={"width": 1440, "height": 1080},
    wait_between_actions=1.0,
    minimum_wait_page_load_time=0.5,
    wait_for_network_idle_page_load_time=0.8,
    highlight_elements=False,
)

browser_session = BrowserSession(browser_profile=browser_profile)

llm = ChatOpenAI(
    model=MODEL_NAME,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    temperature=0,
    max_completion_tokens=512,
    reasoning_effort="low",
    add_schema_to_system_prompt=True,
    dont_force_structured_output=True,
    remove_min_items_from_schema=True,
    remove_defaults_from_schema=True,
)

async def run_task(task: str, max_steps: int = 20):
    agent = Agent(
        task=task,
        llm=llm,
        browser_session=browser_session,
        browser_profile=browser_profile,
        use_vision=USE_VISION,
        flash_mode=FLASH_MODE,
        max_history_items=MAX_HISTORY_ITEMS,
        enable_planning=False,
        directly_open_url=True,
        include_recent_events=True,
        llm_timeout=120,
        step_timeout=180,
        max_actions_per_step=5,
        use_thinking=False,
    )
    history = await agent.run(max_steps=max_steps)
    print(json.dumps({
        "done": history.is_done(),
        "successful": history.is_successful(),
        "steps": history.number_of_steps(),
    }, indent=2))
    try:
        print("final_result:")
        print(history.final_result())
    except Exception as exc:
        print(f"final_result unavailable: {exc}")
    return history


In [ ]:
# Cell 3: Edit TASK and run the browser-use agent.
TASK = "Go to https://duckduckgo.com to visit hostinger main webpage, then from there, navigate your way through it to find vps pricing."
history = await run_task(TASK, max_steps=12)
history
